In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
# Advanced AQUA
# Amitava Chakraborty 04 Feb, 2020
#
# Qiskit Aqua and qiskit.finance were removed. The eigensolvers now come from qiskit-algorithms,
# and the small finance helpers this notebook uses (random price data and the portfolio
# Hamiltonian) are written out below with qiskit-optimization.

from qiskit_backend_config import BasicAer
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.applications import OptimizationApplication
import numpy as np
import datetime

sample_most_likely = OptimizationApplication.sample_most_likely


class RandomDataProvider:
    """Random-walk stock prices (stand-in for qiskit.finance.data_providers.RandomDataProvider)."""

    def __init__(self, tickers, start, end, seed=None):
        self._tickers, self._start, self._end = tickers, start, end
        self._rng = np.random.default_rng(seed)
        self._data = []

    def run(self):
        days = (self._end - self._start).days
        self._data = [self._rng.uniform(50, 150) + np.cumsum(self._rng.standard_normal(days))
                      for _ in self._tickers]

    def _returns(self):
        prices = np.array(self._data)
        return (prices[:, 1:] - prices[:, :-1]) / prices[:, :-1]

    def get_period_return_mean_vector(self):
        return self._returns().mean(axis=1)

    def get_period_return_covariance_matrix(self):
        return np.cov(self._returns())


class portfolio:
    """Portfolio optimisation:  min  q x^T Sigma x - mu^T x + penalty (1^T x - budget)^2,  x binary."""

    @staticmethod
    def get_operator(mu, sigma, q, budget, penalty):
        n = len(mu)
        qp = QuadraticProgram()
        for i in range(n):
            qp.binary_var(f'x{i}')
        qp.minimize(constant=penalty * budget ** 2,
                    linear=-np.asarray(mu) - 2 * penalty * budget,
                    quadratic=q * np.asarray(sigma) + penalty * np.ones((n, n)))
        return qp.to_ising()

    @staticmethod
    def portfolio_value(x, mu, sigma, q, budget, penalty):
        return q * np.dot(x, np.dot(sigma, x)) - np.dot(mu, x) + penalty * pow(sum(x) - budget, 2)

from qiskit_backend_config import IBMQ
provider = IBMQ.load_account()


In [ ]:
# set number of assets (= number of qubits)
num_assets = 4

# Generate expected return and covariance matrix from (random) time-series
stocks = [("TICKER%s" % i) for i in range(num_assets)]
print(stocks)

data = RandomDataProvider(tickers=stocks,
                 start=datetime.datetime(2016,1,1),
                 end=datetime.datetime(2016,1,30),
                 seed=123)
data.run()

print(data._data)
mu = data.get_period_return_mean_vector()
sigma = data.get_period_return_covariance_matrix()
print(mu)
print(sigma)

In [ ]:
q = 0.1 # set risk factor
budget = int(num_assets / 2) # set budget
penalty = num_assets # set parameter to scale the budget penalty term

qubitOp, offset = portfolio.get_operator(mu, sigma, q, budget, penalty)
print(qubitOp)
print(offset)

In [ ]:
def index_to_selection(i, num_assets):
    s = "{0:b}".format(i).rjust(num_assets)
    x = np.array([1 if s[i]=='1' else 0 for i in reversed(range(num_assets))])
    return x

def print_result(result):
    selection = sample_most_likely(result['eigvecs'][0])
    value = portfolio.portfolio_value(selection, mu, sigma, q, budget, penalty)
    print('Optimal: selection {}, value {:.4f}'.format(selection, value))

    probabilities = np.abs(result['eigvecs'][0])**2
    i_sorted = reversed(np.argsort(probabilities))
    print('\n----------------- Full result ---------------------')
    print('selection\tvalue\t\tprobability')
    print('---------------------------------------------------')
    for i in i_sorted:
        x = index_to_selection(i, num_assets)
        value = portfolio.portfolio_value(x, mu, sigma, q, budget, penalty)    
        probability = probabilities[i]
        print('%10s\t%.4f\t\t%.4f' %(x, value, probability))

In [ ]:
exact_eigensolver = NumPyMinimumEigensolver()
exact_result = exact_eigensolver.compute_minimum_eigenvalue(qubitOp)
result = {'eigvecs': [exact_result.eigenstate.data]}

In [ ]:
print(result['eigvecs'][0])
print_result(result)
plot_bloch_multivector(result['eigvecs'][0])

In [ ]:
backend = BasicAer.get_backend('statevector_simulator')
seed = 50
np.random.seed(seed)

cobyla = COBYLA()
cobyla.set_options(maxiter=500)
ry = TwoLocal(qubitOp.num_qubits, 'ry', 'cz', reps=3, entanglement='full')   # Aqua's RY variational form
vqe = VQE(StatevectorEstimator(seed=seed), ry, cobyla,
          initial_point=np.random.uniform(-np.pi, np.pi, ry.num_parameters))

vqe_result = vqe.compute_minimum_eigenvalue(qubitOp)
result = {'eigvecs': [Statevector(ry.assign_parameters(vqe_result.optimal_parameters)).data]}

print_result(result)
